In [1]:
from langgraph.graph import StateGraph , START , END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict , Annotated
from pydantic import BaseModel , Field
import operator

In [2]:
load_dotenv()

True

In [3]:
model = ChatOpenAI(model = 'gpt-4o-mini')
# this model gives structured output 

# requirement is structured output 
# 1. Feedback 2. Score

In [4]:
class EvaluationSchema(BaseModel):
    feedback : str = Field(description = 'Detailed feedback for the essay')
    score : int = Field(description = 'Score out of 10' , ge = 0 , le = 10)

In [5]:
structured_model = model.with_structured_output(EvaluationSchema)

In [6]:
essay = """**Role of India in Artificial Intelligence (AI)**

India is rapidly emerging as a global force in the field of Artificial Intelligence (AI). With its vast pool of technical talent, expanding digital infrastructure, and strong governmental support, the country is positioning itself as a key player in shaping the future of AI. The role of India in AI is multifaceted, spanning innovation, economic growth, governance, and societal transformation.

India has one of the largest populations of software engineers and data scientists in the world. Prestigious institutions produce highly skilled graduates who contribute to AI research and development globally. Additionally, India’s well-established IT sector plays a crucial role in implementing AI solutions across industries, making the country a major contributor to global technology services.

The Indian government has taken significant steps to promote AI adoption through various initiatives. These programs aim to integrate AI into sectors such as healthcare, agriculture, education, and smart cities. The focus is not only on technological advancement but also on ensuring inclusive and ethical AI development that benefits all sections of society.

AI is transforming multiple sectors in India. In healthcare, it helps in early diagnosis and improving access to medical services. In agriculture, it assists farmers with crop prediction and soil analysis. In education, AI enables personalized learning experiences, while in urban development, it supports efficient traffic and waste management systems. These applications are especially important in a diverse and populous country like India.

India also has a rapidly growing startup ecosystem, with many companies working on advanced AI technologies such as machine learning, natural language processing, and computer vision. Major cities have become innovation hubs, attracting investment and fostering entrepreneurship.

However, India faces challenges such as data privacy concerns, lack of high-quality datasets, and a digital divide between urban and rural areas. Addressing these challenges will be crucial for sustainable AI growth. At the same time, these issues present opportunities for innovation and policy improvement.

In conclusion, India stands at a crucial point in the AI revolution. By leveraging its strengths in talent, infrastructure, and policy, the country has the potential to become a global leader in AI. The future of AI in India lies not only in technological progress but also in its ability to create inclusive and impactful solutions that improve lives.
"""

In [7]:
prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}' 
structured_model.invoke(prompt)

EvaluationSchema(feedback="The essay presents a structured and coherent argument about India's role in Artificial Intelligence. The introduction sets the stage effectively, outlining the key themes that will be discussed, such as innovation, economic growth, governance, and societal transformation. The subsequent paragraphs are logically organized, flowing from the strengths of India's talent pool and government initiatives to the specific applications of AI across various sectors.\n\nThe language used is formal and appropriate for an academic setting, showcasing a good command of vocabulary and syntax. The use of transition words and phrases aids in maintaining clarity and coherence throughout the essay.\n\nHowever, the essay could benefit from more specific examples and data to support the claims made, especially regarding India's challenges and the specific initiatives undertaken by the government. Additionally, while the conclusion summarizes the key points well, it could be enhanc

In [8]:
class UPSCState(TypedDict):
    essay : str
    language_feedback : str
    analysis_feedback : str
    clarity_feedback : str
    overall_feedback : str
    individual_scores : Annotated[list[int],operator.add]
    avg_score : float

# reducer function used here is add , which will add all the individual_score to that list 

In [9]:
def evaluate_language(state : UPSCState) ->  UPSCState:

    prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}'
    output = structured_model.invoke(prompt)

    return {'language_feedback' : output.feedback , 'individual_score':[output.score]}

In [10]:
def evaluate_analysis(state : UPSCState) ->  UPSCState:

    prompt = f'Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}'
    output = structured_model.invoke(prompt)

    return {'analysis_feedback' : output.feedback , 'individual_scores':[output.score]}

In [11]:
def evaluate_thought(state : UPSCState) ->  UPSCState:

    prompt = f'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}'
    output = structured_model.invoke(prompt)

    return {'clarity_feedback' : output.feedback , 'individual_scores':[output.score]}

In [12]:
def final_evaluation(state : UPSCState) -> UPSCState : 
    # summary 
    prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state["language_feedback"]} \
    \n depth of analysis feedback - {state["analysis_feedback"]} \n clarity of thought feedback - {state["clarity_feedback"]}'

    # we will use the normal model here as no requirement of getting score particularly
    overall_feedback = model.invoke(prompt).content
    
    # average calculate
    avg_score = sum(state['individual_scores'])/len(state['individual_scores'])

    return {'overall_feedback':overall_feedback , 'avg_score' : avg_score}
    

In [13]:
graph = StateGraph(UPSCState)

In [14]:
graph.add_node('evaluate_language' ,evaluate_language)
graph.add_node('evaluate_analysis' ,evaluate_analysis)
graph.add_node('evaluate_thought' ,evaluate_thought)
graph.add_node('final_evaluation' ,final_evaluation)

In [15]:
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation', END)


In [16]:
workflow = graph.compile()

In [17]:
initial_state = {
    'essay' : essay
}

final_state =  workflow.invoke(initial_state)
print(final_state)


{'essay': '**Role of India in Artificial Intelligence (AI)**\n\nIndia is rapidly emerging as a global force in the field of Artificial Intelligence (AI). With its vast pool of technical talent, expanding digital infrastructure, and strong governmental support, the country is positioning itself as a key player in shaping the future of AI. The role of India in AI is multifaceted, spanning innovation, economic growth, governance, and societal transformation.\n\nIndia has one of the largest populations of software engineers and data scientists in the world. Prestigious institutions produce highly skilled graduates who contribute to AI research and development globally. Additionally, India’s well-established IT sector plays a crucial role in implementing AI solutions across industries, making the country a major contributor to global technology services.\n\nThe Indian government has taken significant steps to promote AI adoption through various initiatives. These programs aim to integrate A

In [18]:
final_state['avg_score']

7.5

In [19]:
final_state['individual_scores']

[7, 8]